In [36]:
import pandas as pd

In [37]:
df=pd.read_csv('Rajshahi.csv')

In [38]:
df.head()

,year,rhum_P1000,rhum_P300,rhum_P500,rhum_P850,shum_P1000,shum_P300,shum_P500,shum_P850,temp_P1000,...,vvel_P850,vwnd_P1000,vwnd_P300,vwnd_P500,vwnd_P850,zpt_P1000,zpt_P300,zpt_P500,zpt_P850,hwds
0,1940,50.137894,39.577994,36.561080,61.393132,0.013008,0.000211,0.001390,0.009616,30.141215,...,-0.065976,0.882851,3.406074,-0.958321,1.318192,400.402815,93507.417373,56699.386410,14352.962813,13
1,1941,51.840665,30.333821,30.284458,61.655236,0.013476,0.000116,0.001072,0.010191,30.318966,...,-0.003804,0.553034,3.184018,-0.726098,1.910098,450.833216,93458.725091,56776.145578,14425.832938,3
2,1942,39.952779,28.368768,26.827284,58.737453,0.011927,0.000148,0.001096,0.010164,32.860602,...,0.003180,0.580241,1.467711,-1.024361,1.725394,349.409644,93695.746174,56874.045043,14408.149551,23
3,1943,57.140825,33.994090,30.470804,69.836630,0.014305,0.000155,0.001184,0.010855,29.419244,...,-0.101532,1.091553,3.036690,-2.104868,4.054800,389.356990,93518.181965,56733.122155,14331.844880,0
4,1944,33.346496,34.924894,27.167101,54.755049,0.010605,0.000140,0.001018,0.009449,33.392273,...,-0.084855,0.573812,0.482867,-1.137200,1.871743,414.667838,93657.525302,56849.608263,14472.691729,35


In [39]:
df.shape

(86, 30)

In [40]:
df["rhum_P1000"].mean()

np.float64(45.67443826760478)

In [41]:
# Sort by year first
df = df.sort_values("year")

# Features and target
X = df.drop(columns=["year", "hwds"])
y = df["hwds"]

# Create masks
train_mask = df["year"] <= 2008
test_mask = df["year"] > 2008

# Split
X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

print("Training years:", df.loc[train_mask, "year"].min(), "-", df.loc[train_mask, "year"].max())
print("Testing years :", df.loc[test_mask, "year"].min(), "-", df.loc[test_mask, "year"].max())

Training years: 1940 - 2008
Testing years : 2009 - 2025


In [42]:
# ==========================================
# Scale raw 28 train -  test  Features
# ==========================================


from sklearn.preprocessing import StandardScaler
import pandas as pd

scaler = StandardScaler()

# Scale training data
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

# Scale test data
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print(X_train_scaled.head())


   rhum_P1000  rhum_P300  rhum_P500  rhum_P850  shum_P1000  shum_P300  \
0    0.562926   1.101487   1.903532   0.050596    0.278804   1.051812   
1    0.742602  -0.067931   0.658247   0.085461    0.565596  -1.058336   
2   -0.511804  -0.316517  -0.027658  -0.302668   -0.384176  -0.349524   
3    1.301873   0.395105   0.695218   1.173767    1.074103  -0.192944   
4   -1.208897   0.512855   0.039762  -0.832416   -1.195277  -0.519536   

   shum_P500  shum_P850  temp_P1000  temp_P300  ...  vvel_P500  vvel_P850  \
0   1.380214  -0.599174   -0.867392  -1.188253  ...  -0.426189   0.359879   
1  -0.169059   0.030860   -0.769341  -1.782285  ...   0.123987   1.693428   
2  -0.054846   0.001121    0.632674  -1.017558  ...  -1.051471   1.843222   
3   0.374445   0.759408   -1.265644  -1.766942  ...  -1.884825  -0.402764   
4  -0.437557  -0.782813    0.925954  -1.777198  ...   0.978375  -0.045051   

   vwnd_P1000  vwnd_P300  vwnd_P500  vwnd_P850  zpt_P1000  zpt_P300  zpt_P500  \
0    0.002376   0

In [43]:
!pip install feature-engine


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
from feature_engine.selection import MRMR

In [45]:
# MIQ Feature Selection
selector = MRMR(
    method="MIQ",
    regression=True,
    max_features=20,
    random_state=42
)

# Fit only on training data
selector.fit(X_train, y_train)

# Transform both training and testing data
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

print("Selected Features:")
print(X_train_selected.columns.tolist())

print("Training Shape:", X_train_selected.shape)
print("Testing Shape :", X_test_selected.shape)

C:\Users\User\anaconda3\Lib\site-packages\feature_engine\selection\mrmr.py:465: RuntimeWarning: divide by zero encountered in divide
  mrmr = relevance / redundance
C:\Users\User\anaconda3\Lib\site-packages\feature_engine\selection\mrmr.py:465: RuntimeWarning: divide by zero encountered in divide
  mrmr = relevance / redundance
C:\Users\User\anaconda3\Lib\site-packages\feature_engine\selection\mrmr.py:465: RuntimeWarning: divide by zero encountered in divide
  mrmr = relevance / redundance


Selected Features:
['rhum_P1000', 'rhum_P850', 'shum_P1000', 'shum_P300', 'shum_P850', 'temp_P1000', 'temp_P300', 'temp_P500', 'temp_P850', 'uwnd_P1000', 'uwnd_P500', 'uwnd_P850', 'vvel_P300', 'vvel_P850', 'vwnd_P1000', 'vwnd_P300', 'vwnd_P850', 'zpt_P1000', 'zpt_P500', 'zpt_P850']
Training Shape: (69, 20)
Testing Shape : (17, 20)


In [46]:
print(X_train_selected)

    rhum_P1000  rhum_P850  shum_P1000  shum_P300  shum_P850  temp_P1000  \
0    50.137894  61.393132    0.013008   0.000211   0.009616   30.141215   
1    51.840665  61.655236    0.013476   0.000116   0.010191   30.318966   
2    39.952779  58.737453    0.011927   0.000148   0.010164   32.860602   
3    57.140825  69.836630    0.014305   0.000155   0.010855   29.419244   
4    33.346496  54.755049    0.010605   0.000140   0.009449   33.392273   
..         ...        ...         ...        ...        ...         ...   
64   48.299269  67.446063    0.014080   0.000171   0.011540   31.982877   
65   48.825745  67.976289    0.013944   0.000153   0.011228   31.494822   
66   47.712824  66.446721    0.013492   0.000179   0.010807   31.359947   
67   44.651315  68.083230    0.013059   0.000132   0.011080   31.610072   
68   46.282303  68.289598    0.013816   0.000173   0.011395   32.133904   

    temp_P300  temp_P500  temp_P850  uwnd_P1000  uwnd_P500  uwnd_P850  \
0  -35.110257  -9.235206  

In [47]:
print(X_test_selected)

    rhum_P1000  rhum_P850  shum_P1000  shum_P300  shum_P850  temp_P1000  \
69   42.355937  60.899632    0.012469   0.000164   0.010192   32.296781   
70   45.568105  64.096924    0.014395   0.000186   0.011879   33.446873   
71   52.983133  66.698539    0.014421   0.000125   0.010849   30.494497   
72   42.828810  61.734070    0.012710   0.000145   0.010368   32.104683   
73   48.302320  63.443981    0.013192   0.000152   0.010499   30.961507   
74   38.628165  59.989422    0.011451   0.000190   0.009831   32.361006   
75   56.296056  68.160104    0.014860   0.000182   0.010813   29.705644   
76   51.573705  67.131012    0.014629   0.000207   0.011355   31.580998   
77   58.084751  70.827668    0.015170   0.000151   0.011375   29.653228   
78   56.422960  69.912797    0.014790   0.000159   0.011014   29.870804   
79   52.910555  67.024473    0.014548   0.000129   0.010931   30.493481   
80   55.091878  69.246801    0.014004   0.000176   0.010516   29.465687   
81   46.837292  65.156177

In [48]:
# ==========================================
# Scale top 10 selected   Features
# ==========================================

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Fit only on training data
X_train_selected_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_selected),
    columns=X_train_selected.columns,
    index=X_train_selected.index
)

# Transform test data
X_test_selected_scaled = pd.DataFrame(
    scaler.transform(X_test_selected),
    columns=X_test_selected.columns,
    index=X_test_selected.index
)

print("Training Shape:", X_train_selected_scaled.shape)
print("Testing Shape :", X_test_selected_scaled.shape)

print(X_train_selected_scaled.head())

Training Shape: (69, 20)
Testing Shape : (17, 20)
   rhum_P1000  rhum_P850  shum_P1000  shum_P300  shum_P850  temp_P1000  \
0    0.562926   0.050596    0.278804   1.051812  -0.599174   -0.867392   
1    0.742602   0.085461    0.565596  -1.058336   0.030860   -0.769341   
2   -0.511804  -0.302668   -0.384176  -0.349524   0.001121    0.632674   
3    1.301873   1.173767    1.074103  -0.192944   0.759408   -1.265644   
4   -1.208897  -0.832416   -1.195277  -0.519536  -0.782813    0.925954   

   temp_P300  temp_P500  temp_P850  uwnd_P1000  uwnd_P500  uwnd_P850  \
0  -1.188253  -1.764404  -0.909043   -0.576767   0.372731   0.892792   
1  -1.782285  -2.492785  -0.291076   -0.894446  -0.102518   0.217257   
2  -1.017558  -1.094161   0.643040   -0.119369  -1.493022  -0.778186   
3  -1.766942  -1.315334  -1.047323   -2.275820  -1.612581  -0.996710   
4  -1.777198  -1.368996   0.345293    0.624954  -0.643298  -0.948331   

   vvel_P300  vvel_P850  vwnd_P1000  vwnd_P300  vwnd_P850  zpt_P1000  \


In [49]:
# ==========================================
# apply PCA on 15 selected feature    features 
# ==========================================

from sklearn.decomposition import PCA
# Create PCA object
pca = PCA(n_components=0.95,random_state=42)

# Fit PCA on training data and transform
X_train_pca = pca.fit_transform(X_train_selected_scaled)

# Transform test data
X_test_pca = pca.transform(X_test_selected_scaled)

In [50]:
print("Original number of features :", X_train_scaled.shape[1])
print("Number of principal components :", X_train_pca.shape[1])

Original number of features : 28
Number of principal components : 9


In [51]:
# Convert selected pca  to DataFrame
import pandas as pd

pc_names = [f"PC{i+1}" for i in range(pca.n_components_)]

X_train_pca = pd.DataFrame(
    X_train_pca,
    columns=pc_names,
    index=X_train_scaled.index
)

X_test_pca = pd.DataFrame(
    X_test_pca,
    columns=pc_names,
    index=X_test_scaled.index
)

print("\nTraining Shape:", X_train_pca.shape)
print("Testing Shape :", X_test_pca.shape)
 
print(X_train_pca.head())
print(X_test_pca.head())


Training Shape: (69, 9)
Testing Shape : (17, 9)
        PC1       PC2       PC3       PC4       PC5       PC6       PC7  \
0  1.167884 -3.416656 -0.459320 -0.568106  1.086150 -1.646591  0.234278   
1  1.216286 -3.073984  1.039358 -1.915887 -0.347463  0.581254  0.504874   
2 -0.300809 -1.563088 -0.778291 -0.750582 -0.715933  2.120500  0.203654   
3  4.430082 -0.493144 -1.502975 -2.737216 -2.836081 -2.043318 -0.086672   
4 -2.038204 -2.042431  0.549884 -0.750442 -2.127510  1.395713 -0.357270   

        PC8       PC9  
0  0.301457 -0.276911  
1 -1.163461  0.849717  
2  0.321917  1.135985  
3 -1.053490  1.332658  
4  0.261603  0.055872  
         PC1       PC2       PC3       PC4       PC5       PC6       PC7  \
69  0.189098  1.309644 -0.322215  0.641874 -0.472048  1.087758  0.407069   
70 -0.511017  5.361000  1.147291  2.009191 -1.085036 -1.827013 -0.928895   
71  2.711361  0.249763  0.593402  0.599064  0.684378  1.030196  1.575207   
72 -0.461121  1.470898  0.753333  0.234912  0.400887

In [52]:
# =====================================================
# Model 5 - Linear Regression
# Scaling is NOT required.
# =====================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Linear Regression Model
# =====================================================

model = LinearRegression()

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Linear Regression")

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using 28 raw features\n")
print(results)

Linear Regression

Train using 28 raw features

               Model       MAE      RMSE        R2        CC
0  Linear Regression  6.908782  8.683836  0.591857  0.814878


In [53]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Linear Regression Model
# =====================================================

model = LinearRegression()

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Linear Regression")

# Train
model.fit(X_train_selected, y_train)

# Predict
y_pred = model.predict(X_test_selected)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using 10 selected  features\n")
print(results)

Linear Regression

Train using 10 selected  features

               Model       MAE      RMSE        R2        CC
0  Linear Regression  7.281741  9.246474  0.537255  0.847522


In [54]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Linear Regression Model
# =====================================================

model = LinearRegression()

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Linear Regression")

# Train
model.fit(X_train_pca, y_train)

# Predict
y_pred = model.predict(X_test_pca)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using pca features\n")
print(results)

Linear Regression

Train using pca features

               Model       MAE     RMSE        R2        CC
0  Linear Regression  6.186289  7.58308  0.688771  0.892917


In [55]:
# =====================================================
# Model 6 - Ridge Regression
# Scaling is REQUIRED.
# =====================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Ridge Regression Model
# =====================================================

model = Ridge(
    alpha=1.0,
    random_state=42
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Ridge Regression")

# Train
model.fit(X_train_scaled, y_train)

# Predict
y_pred = model.predict(X_test_scaled)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Ridge Regression"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using 28 raw features\n")
print(results)

Ridge Regression

Train using 28 raw features

              Model       MAE      RMSE       R2        CC
0  Ridge Regression  6.084164  7.949575  0.65796  0.860398


In [68]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Ridge Regression Model
# =====================================================

model = Ridge(
    alpha=1.0,
    random_state=42
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Ridge Regression")

# Train
model.fit(X_train_selected_scaled, y_train)

# Predict
y_pred = model.predict(X_test_selected_scaled)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Ridge Regression"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using 20 selected features\n")
print(results)

Ridge Regression

Train using 20 selected features

              Model       MAE     RMSE        R2        CC
0  Ridge Regression  7.593968  9.11999  0.549829  0.845329


In [57]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Ridge Regression Model
# =====================================================

model = Ridge(
    alpha=1.0,
    random_state=42
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Ridge Regression")

# Train
model.fit(X_train_pca, y_train)

# Predict
y_pred = model.predict(X_test_pca)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Ridge Regression"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using pca features\n")
print(results)

Ridge Regression

Train using pca features

              Model       MAE      RMSE        R2        CC
0  Ridge Regression  6.186827  7.578633  0.689136  0.893634


In [58]:
# =====================================================
# Model 7 - Lasso Regression
# Scaling is REQUIRED.
# =====================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import Lasso

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Lasso Model
# =====================================================

model = Lasso(
    alpha=0.01,
    max_iter=10000,
    random_state=42
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Lasso Regression")

# Train
model.fit(X_train_scaled, y_train)

# Predict
y_pred = model.predict(X_test_scaled)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Lasso Regression"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using 28 raw features\n")
print(results)

Lasso Regression

Train using 28 raw features

              Model       MAE      RMSE        R2        CC
0  Lasso Regression  6.335506  8.301735  0.626985  0.836148


In [59]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Lasso

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Lasso Model
# =====================================================

model = Lasso(
    alpha=0.01,
    max_iter=10000,
    random_state=42
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Lasso Regression")

# Train
model.fit(X_train_selected_scaled, y_train)

# Predict
y_pred = model.predict(X_test_selected_scaled)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Lasso Regression"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using 10 selected features\n")
print(results)

Lasso Regression

Train using 10 selected features

              Model       MAE      RMSE        R2        CC
0  Lasso Regression  7.584717  9.145959  0.547261  0.845974


In [60]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Lasso

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Lasso Model
# =====================================================

model = Lasso(
    alpha=0.01,
    max_iter=10000,
    random_state=42
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Lasso Regression")

# Train
model.fit(X_train_pca, y_train)

# Predict
y_pred = model.predict(X_test_pca)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Lasso Regression"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using pca features\n")
print(results)

Lasso Regression

Train using pca features

              Model       MAE      RMSE        R2        CC
0  Lasso Regression  6.190023  7.585717  0.688554  0.893138


In [61]:
# =====================================================
# Model 8 - Elastic Net Regression
# Scaling is REQUIRED.
# =====================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import ElasticNet

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Elastic Net Model
# =====================================================

model = ElasticNet(
    alpha=0.01,
    l1_ratio=0.5,
    max_iter=10000,
    random_state=42
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Elastic Net Regression")

# Train
model.fit(X_train_scaled, y_train)

# Predict
y_pred = model.predict(X_test_scaled)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Elastic Net"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using 28 raw features\n")
print(results)

Elastic Net Regression

Train using 28 raw features

         Model       MAE      RMSE        R2        CC
0  Elastic Net  6.108022  8.040093  0.650126  0.852728


In [62]:
import numpy as np
import pandas as pd

from sklearn.linear_model import ElasticNet

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Elastic Net Model
# =====================================================

model = ElasticNet(
    alpha=0.01,
    l1_ratio=0.5,
    max_iter=10000,
    random_state=42
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Elastic Net Regression")

# Train
model.fit(X_train_selected_scaled, y_train)

# Predict
y_pred = model.predict(X_test_selected_scaled)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Elastic Net"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using 10 selected features\n")
print(results)

Elastic Net Regression

Train using 10 selected features

         Model       MAE      RMSE        R2        CC
0  Elastic Net  7.626341  9.163522  0.545521  0.844441


In [63]:
import numpy as np
import pandas as pd

from sklearn.linear_model import ElasticNet

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Elastic Net Model
# =====================================================

model = ElasticNet(
    alpha=0.01,
    l1_ratio=0.5,
    max_iter=10000,
    random_state=42
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Elastic Net Regression")

# Train
model.fit(X_train_pca, y_train)

# Predict
y_pred = model.predict(X_test_pca)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Elastic Net"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using pca features\n")
print(results)

Elastic Net Regression

Train using pca features

         Model       MAE      RMSE        R2        CC
0  Elastic Net  6.188351  7.582815  0.688792  0.893277


In [64]:
# =====================================================
# Model - Gaussian Process Regression (GPR)
# Scaling is REQUIRED.
# =====================================================

import numpy as np
import pandas as pd

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Gaussian Process Regression Model
# =====================================================

kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2))
    + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-6, 1e1))
)

model = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Gaussian Process Regression")

# Train
model.fit(X_train_scaled, y_train)

# Predict
y_pred = model.predict(X_test_scaled)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Gaussian Process Regression"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using 28 raw features\n")
print(results)

Gaussian Process Regression

Train using 28 raw features

                         Model       MAE      RMSE        R2        CC
0  Gaussian Process Regression  5.982753  7.640653  0.684027  0.882461


In [65]:
import numpy as np
import pandas as pd

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Gaussian Process Regression Model
# =====================================================

kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2))
    + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-6, 1e1))
)

model = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Gaussian Process Regression")

# Train
model.fit(X_train_selected_scaled, y_train)

# Predict
y_pred = model.predict(X_test_selected_scaled)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Gaussian Process Regression"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using 10 selected  features\n")
print(results)


Gaussian Process Regression

Train using 10 selected  features

                         Model       MAE      RMSE        R2        CC
0  Gaussian Process Regression  6.985705  8.298506  0.627275  0.871101


In [66]:
import numpy as np
import pandas as pd

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# Define Gaussian Process Regression Model
# =====================================================

kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2))
    + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-6, 1e1))
)

model = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cc = np.corrcoef(y_true, y_pred)[0, 1]

    return mae, rmse, r2, cc

# =====================================================
# Train Model
# =====================================================

print("=" * 60)
print("Gaussian Process Regression")

# Train
model.fit(X_train_pca, y_train)

# Predict
y_pred = model.predict(X_test_pca)

# =====================================================
# Evaluation
# =====================================================

mae, rmse, r2, cc = evaluate(y_test, y_pred)

# =====================================================
# Results
# =====================================================

results = pd.DataFrame({
    "Model": ["Gaussian Process Regression"],
    "MAE": [mae],
    "RMSE": [rmse],
    "R2": [r2],
    "CC": [cc]
})

print("\nTrain using pca  features\n")
print(results)


Gaussian Process Regression

Train using pca  features

                         Model       MAE      RMSE        R2        CC
0  Gaussian Process Regression  6.246403  7.512401  0.694545  0.897255


In [67]:
# ============================================================
# Multi Layer Perceptron Regression (scikit-learn)
# ============================================================

import numpy as np
import pandas as pd

from sklearn.neural_network import MLPRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from scipy.stats import pearsonr

# ------------------------------------------------------------
# Create MLP Model
# ------------------------------------------------------------

mlp = MLPRegressor(

    hidden_layer_sizes=(8,),      # One hidden layer with 8 neurons
    activation='relu',
    solver='adam',

    alpha=0.001,                  # L2 Regularization
    batch_size=8,

    learning_rate='adaptive',
    learning_rate_init=0.001,

    max_iter=5000,

    early_stopping=True,
    validation_fraction=0.15,

    n_iter_no_change=30,

    random_state=42
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

mlp.fit(X_train_pca, y_train)

# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------

y_pred = mlp.predict(X_test_pca)

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

MAE = mean_absolute_error(y_test, y_pred)

RMSE = np.sqrt(mean_squared_error(y_test, y_pred))

R2 = r2_score(y_test, y_pred)

CC = pearsonr(y_test, y_pred)[0]

# ------------------------------------------------------------
# Results Table
# ------------------------------------------------------------

results = pd.DataFrame({

    "Model":["Multi Layer Perceptron"],
    "MAE":[MAE],
    "RMSE":[RMSE],
    "R2":[R2],
    "CC":[CC]

})

print(results)

                    Model       MAE      RMSE        R2        CC
0  Multi Layer Perceptron  8.610079  9.960788  0.462997  0.769298
